## Milestone 1 (Function-level Profiling) - Lecture 3

In [14]:
"""
Mandelbrot Set Generator
Author : [ Mathias Jørgensen ]
Course : Numerical Scientific Computing 2026
"""
import numpy as np
import matplotlib.pyplot as plt
import time, statistics
import cProfile, pstats
from mandelbrot_3 import compute_mandelbrot_naive, compute_mandelbrot_numpy

In [15]:
# Parameters
x_dim = (-2, 1)
y_dim = (-1.5, 1.5)
resx = 512
resy = 512

#Using cProfile on both naive and numpy function
cProfile.run('compute_mandelbrot_naive(x_dim, y_dim, resx, resy)',
                'naive_profile.prof')
cProfile.run('compute_mandelbrot_numpy(x_dim, y_dim, resx, resy)',
                'numpy_profile.prof')

for name in ('naive_profile.prof', 'numpy_profile.prof'):
    stats = pstats.Stats(name)
    stats.sort_stats('cumulative')
    stats.print_stats(10)


Thu Feb 26 14:50:33 2026    naive_profile.prof

         5743705 function calls in 6.616 seconds

   Ordered by: cumulative time
   List reduced from 20 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    6.616    6.616 {built-in method builtins.exec}
        1    0.000    0.000    6.616    6.616 <string>:1(<module>)
        1    1.282    1.282    6.615    6.615 c:\UNI\8.Semester\numerical_scientific_computing\mandelbrot-nsc\mandelbrot_3.py:39(compute_mandelbrot_naive)
   262144    4.441    0.000    5.328    0.000 c:\UNI\8.Semester\numerical_scientific_computing\mandelbrot-nsc\mandelbrot_3.py:26(mandelbrot_point_naive)
  5481530    0.888    0.000    0.888    0.000 {built-in method builtins.abs}
        2    0.004    0.002    0.005    0.003 c:\Users\mathi\miniforge3\envs\nsc2026\Lib\site-packages\numpy\_core\function_base.py:26(linspace)
        2    0.001    0.001    0.001    0.001 c:\Users\mathi\miniforg

Questions to answer:

1. Which function takes most total time?
    mandelbrot_point_naive for the naive approach.
    Compute_mandelbrot_numpy for the numpy approach.

2. Are there functions called surprisingly many times?
    No only in naive approach it has called mandelbrot_point_naive many times, but that is expectable.

3. How does the NumPy profile compare to naive?
    Faster, less function calls and less overall time.

4. Where does NumPy spend its time?
    Almost all time is spend inside the function compute_mandelbrot_numpy


## Milestone 2 (Line-level Profiling) - Lecture 3 
Wrote profile results to 'mandelbrot_3.py.lprof'
Timer unit: 1e-06 s

Total time: 10.1513 s
File: .\mandelbrot_3.py
Function: compute_mandelbrot_naive at line 26

Line #      Hits         Time  Per Hit   % Time  Line Contents
==============================================================
    26                                           @profile
    27                                           def compute_mandelbrot_naive(x_dim = tuple[float, float],
    28                                                                        y_dim = tuple[float, float],
    29                                                                        res_x = int,
    30                                                                        res_y = int,
    31                                                                        max_iter=100):
    32
    33                                               # Pulling out variables from tuples
    34         1          4.6      4.6      0.0      x_min, x_max = x_dim
    35         1          0.4      0.4      0.0      y_min, y_max = y_dim
    36
    37                                               # Create 1D arrays
    38         1         86.9     86.9      0.0      x = np.linspace(x_min, x_max, res_x)
    39         1         28.6     28.6      0.0      y = np.linspace(y_min, y_max, res_y)
    40         1         21.6     21.6      0.0      result = np.zeros((res_y, res_x), dtype=int)
    41
    42       513        118.9      0.2      0.0      for i in range(res_y):
    43    262656      82268.6      0.3      0.8          for j in range(res_x):
    44    262144    1424432.9      5.4     14.0              c = x[j] + 1j * y[i]
    45    262144      74211.4      0.3      0.7              z = 0
    46   5743674    1510219.7      0.3     14.9              for n in range(max_iter):
    47   5698796    3376748.8      0.6     33.3                  if abs(z) > 2:
    48    217266     197393.7      0.9      1.9                      result[i, j] = n
    49    217266      69766.0      0.3      0.7                      break
    50   5481530    3346907.8      0.6     33.0                  z = z*z + c
    51                                                       else:
    52     44878      69124.8      1.5      0.7                  result[i, j] = max_iter
    53         1          0.8      0.8      0.0      return result

Add a profiling section to your MP1 mini-report. In your own words, answer:
1. cProfile on naive vs NumPy: How many functions appear in each profile? What does this difference tell you about where the work actually happens?
    20 functions appear in naive, while 38 appear in numpy. The work is primarily within the functions and in abs(z) > 2

2. line profiler on naive: Which lines dominate runtime? What fraction of total time is spent in the inner loop?
    Line 47 and line 50, with (if abs(z) > 2) and (z = z*z + c)

3. Based on your profiling results: why is NumPy faster than naive Python?
    The number of function calls are far less for numpy (73 function calls) than in the naive python approach (5743705 function calls)

4. What would you need to change to make the naive version faster? (hint: what does line profiler tell you about the inner loop?)
    Especially change line 47 and 50